In [1]:
import cmocean

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as plt
from matplotlib import pyplot as plt

from scipy.io import loadmat

In [ ]:


data = loadmat(r"C:\Users\rensk\OneDrive\Documenten\PhD\DATA\Mooring\Moorings\TMB\TMBLR000.mat")



In [3]:

data = data  # your ADCP dictionary

# --- 1. Create datetime coordinate ---
timestamps = pd.to_datetime({
    'year': data['SerYear'].flatten() + 2000,
    'month': data['SerMon'].flatten(),
    'day': data['SerDay'].flatten(),
    'hour': data['SerHour'].flatten(),
    'minute': data['SerMin'].flatten(),
    'second': data['SerSec'].flatten(),
    'microsecond': data['SerHund'].flatten() * 10000  # 100th sec -> μs
})

instrument_depth = 452
# --- 2. Create depth coordinate ---
bin_indices = data['SerBins'].flatten()
bin_size = data['RDIBinSize'][0,0]  # meters
bin1_mid = data['RDIBin1Mid'][0,0]  # meters
bin_range = bin1_mid + bin_size * (bin_indices - 1)  # depth of each bin
depths = instrument_depth - bin_range

# --- 3. Function to clean velocity / correlation arrays ---
def clean_array(arr, scale=1000):
    arr = np.array(arr, dtype=float)
    arr[arr == -32768] = np.nan
    return arr / scale  # default mm/s -> m/s

# --- 4. Build xarray Dataset ---
ds = xr.Dataset(
    data_vars={
        # Velocity
        'E': (('ensemble', 'bin'), clean_array(data['SerEmmpersec'])),
        'N': (('ensemble', 'bin'), clean_array(data['SerNmmpersec'])),
        'V': (('ensemble', 'bin'), clean_array(data['SerVmmpersec'])),
        'ER': (('ensemble', 'bin'), clean_array(data['SerErmmpersec'])),
        # Correlation / amplitude
        'E_amp': (('ensemble', 'bin'), data['SerEAAcnt'].astype(float)),
        'C_amp': (('ensemble', 'bin'), data['SerCAcnt'].astype(float)),
        'Mag': (('ensemble', 'bin'), clean_array(data['SerMagmmpersec'])),
        'Dir_deg': (('ensemble', 'bin'), np.array(data['SerDir10thDeg'], dtype=float)/10),
        # Scalars (1D)
        'pitch_deg': (('ensemble',), np.array(data['AnP100thDeg'], dtype=float).flatten()/100),
        'roll_deg': (('ensemble',), np.array(data['AnR100thDeg'], dtype=float).flatten()/100),
        'heading_deg': (('ensemble',), np.array(data['AnH100thDeg'], dtype=float).flatten()/100),
        'temperature_C': (('ensemble',), np.array(data['AnT100thDeg'], dtype=float).flatten()/100),
        'depth_m': (('ensemble',), np.array(data['AnDepthmm'], dtype=float).flatten()/1000),
        'battery': (('ensemble',), np.array(data['AnBatt'], dtype=float).flatten()),
    },
    coords={
        'ensemble': timestamps,
        'bin': depths
    }
)

# --- 5. Add metadata ---
ds.attrs['ADCP_system'] = data['RDISystem'][0]
ds.attrs['file_name'] = data['RDIFileName'][0]

ds


<xarray.Dataset>
Dimensions:        (ensemble: 21123, bin: 119)
Coordinates:
  * ensemble       (ensemble) datetime64[ns] 2023-06-21T13:05:10.006960 ... 2...
  * bin            (bin) float32 438.7 433.7 428.7 ... -141.3 -146.3 -151.3
Data variables: (12/14)
    E              (ensemble, bin) float64 -0.132 -0.674 nan nan ... nan nan nan
    N              (ensemble, bin) float64 0.537 0.289 nan nan ... nan nan nan
    V              (ensemble, bin) float64 0.937 0.083 nan nan ... nan nan nan
    ER             (ensemble, bin) float64 0.178 0.178 nan nan ... nan nan nan
    E_amp          (ensemble, bin) float64 86.0 62.0 53.0 ... 48.0 48.0 48.0
    C_amp          (ensemble, bin) float64 104.0 76.0 56.0 34.0 ... 0.0 0.0 0.0
    ...             ...
    pitch_deg      (ensemble) float64 -42.87 -42.87 -42.87 ... -42.86 -43.19
    roll_deg       (ensemble) float64 46.55 46.55 46.54 ... 46.05 46.06 27.8
    heading_deg    (ensemble) float64 131.5 131.0 128.7 ... 109.7 113.4 146.1
    temperature_C  (ensemble) float64 18.33 18.46 18.46 ... 14.73 14.72 14.74
    depth_m        (ensemble) float64 0.7 0.6 0.6 1.0 0.7 ... 0.6 0.5 0.0 0.4
    battery        (ensemble) float64 165.0 165.0 165.0 ... 137.0 137.0 137.0
Attributes:
    ADCP_system:  Broadband 76.8 kHz
    file_name:    C:\Users\elmerc\NIWA\Ocean Deployments - Projects\TAN2408_k...

In [4]:
import xarray as xr
import numpy as np
import pandas as pd

def clean_adcp(ds, depth_range=None, corr_thresh=50, err_vel_thresh=0.3, flip_bins=True):
    """
    Perform basic ADCP QC and preprocessing.

    Parameters:
    -----------
    ds : xarray.Dataset
        Original ADCP dataset with variables: E, N, V, C_amp, ER, Dir10thDeg, bin, ensemble
    depth_range : tuple (min_depth, max_depth), optional
        Depths to keep (in meters). If None, keep all bins.
    corr_thresh : float
        Minimum correlation amplitude to accept velocity.
    err_vel_thresh : float
        Maximum error velocity (ER) to accept velocity.
    flip_bins : bool
        If True, flips the bin axis so surface is top.

    Returns:
    --------
    ds_clean : xarray.Dataset
        Cleaned dataset with:
        - trimmed depth bins
        - missing ensembles filled (NaN)
        - velocities masked for bad correlation / error
        - horizontal speed (Mag) added
        - directional data in 0–360° (Dir_deg), invalid values masked
        - optional bin flip
    """

    ds_clean = ds.copy()


    # --- Trim depth bins ---
    if depth_range is not None:
        ds_clean = ds_clean.sel(bin=slice(depth_range[0], depth_range[1]))

    # --- Fill missing ensembles ---
    if np.issubdtype(ds_clean['ensemble'].dtype, np.datetime64):
        dt = np.median(np.diff(ds_clean['ensemble'].values))
        full_time = pd.date_range(start=ds_clean['ensemble'].values[0],
                                  end=ds_clean['ensemble'].values[-1],
                                  freq=pd.to_timedelta(dt))
        ds_clean = ds_clean.reindex(ensemble=full_time)
    else:
        full_ens = np.arange(ds_clean['ensemble'].values[0], 
                             ds_clean['ensemble'].values[-1]+1)
        ds_clean = ds_clean.reindex(ensemble=full_ens)

    # --- Mask velocities ---
    for var in ['E', 'N', 'V']:
        if var in ds_clean:
            ds_clean[var] = ds_clean[var].where(ds_clean['C_amp'] > corr_thresh)
            ds_clean[var] = ds_clean[var].where(np.abs(ds_clean['ER']) < err_vel_thresh)

    # --- Horizontal speed ---
    if 'E' in ds_clean and 'N' in ds_clean:
        ds_clean['Mag'] = np.sqrt(ds_clean['E']**2 + ds_clean['N']**2)

   
    if 'Dir_deg' in ds_clean:
        dir_raw = ds_clean['Dir_deg']
        dir_deg = dir_raw.where((dir_raw >= 0) & (dir_raw <= 360))
        ds_clean['Dir_deg'] = dir_deg


            # --- Flip bins if needed ---
        if flip_bins and ds_clean['bin'][0] > ds_clean['bin'][-1]:
            ds_clean = ds_clean.sel(bin=ds_clean['bin'][::-1])


    return ds_clean


In [5]:
ds_clean = clean_adcp(ds, depth_range=None, corr_thresh=50, err_vel_thresh=0.3, flip_bins=True)

# Check bin order and direction
print(ds_clean['bin'].values[:5], ds_clean['bin'].values[-5:])
print(np.nanmin(ds_clean['Dir_deg'].values), np.nanmax(ds_clean['Dir_deg'].values))


[-151.26001 -146.26001 -141.26001 -136.26001 -131.26001] [418.74 423.74 428.74 433.74 438.74]
0.0 360.0


In [ ]:

ds_combined = ds_clean.rename({'bin': 'depth', 'ensemble': 'time'})
ds_combined

<xarray.Dataset>
Dimensions:        (time: 21123, depth: 119)
Coordinates:
  * time           (time) datetime64[ns] 2023-06-21T13:05:10.006960 ... 2024-...
  * depth          (depth) float32 -151.3 -146.3 -141.3 ... 428.7 433.7 438.7
Data variables: (12/14)
    E              (time, depth) float64 nan nan nan nan ... nan nan nan 0.107
    N              (time, depth) float64 nan nan nan nan ... nan nan nan 0.158
    V              (time, depth) float64 nan nan nan nan ... nan nan nan 0.018
    ER             (time, depth) float64 nan nan nan nan ... nan nan nan -0.024
    E_amp          (time, depth) float64 47.0 46.0 47.0 47.0 ... 56.0 78.0 91.0
    C_amp          (time, depth) float64 0.0 0.0 0.0 0.0 ... 37.0 41.0 80.0 80.0
    ...             ...
    pitch_deg      (time) float64 -42.87 -42.87 -42.87 ... -42.86 -42.86 -43.19
    roll_deg       (time) float64 46.55 46.55 46.54 46.55 ... 46.05 46.06 27.8
    heading_deg    (time) float64 131.5 131.0 128.7 130.7 ... 109.7 113.4 146.1
    temperature_C  (time) float64 18.33 18.46 18.46 18.41 ... 14.73 14.72 14.74
    depth_m        (time) float64 0.7 0.6 0.6 1.0 0.7 ... 0.3 0.6 0.5 0.0 0.4
    battery        (time) float64 165.0 165.0 165.0 165.0 ... 137.0 137.0 137.0
Attributes:
    ADCP_system:  Broadband 76.8 kHz
    file_name:    C:\Users\elmerc\NIWA\Ocean Deployments - Projects\TAN2408_k...

In [7]:
ds_combined = ds_combined.sel(depth=ds_combined.depth >= 0)


In [8]:
ds_combined.to_netcdf(r"C:\Users\rensk\OneDrive\Documenten\PhD NZ\DATA\Mooring\Moorings\proc\TMB.nc")
